# Cuál es el problema?

Abandono de clientes (Churn)

Trabajamos con una empresa de telecomunicaciones que quiere ***predecir cuales clientes van a abandonar el servicio*** (churn) para poder tomar acciones preventivas.

Por qué es importante?

* Conseguir un cliente nuevo representa entre 5x y 25x más que retener uno existente.
* Si podemos predecir quien se va, podemos ofrecerle un descuento o llamarlo antes de que cancele.



In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split # Dividir datos en entrenamiento y prueba
from sklearn.compose import ColumnTransformer # Para transformar columnas específicas
from sklearn.preprocessing import OneHotEncoder # Para escalar y codificar variables
from sklearn.linear_model import LogisticRegression # Modelo de regresión logística

# Metricas de evaluación para la clasificación

from sklearn.metrics import (
    confusion_matrix, # Matriz de confusión
    classification_report, # reporte con precisión, recall y f1-score
    accuracy_score, # Porcentaje de predicciones correctas
)

import matplotlib.pyplot as plt #Graficas

print("Librerías importadas correctamente")




In [ ]:
# cargar el archivo CSV en un dataframe de pandas
df = pd.read_csv("abandono_clientes.csv")

#Mostrar las primeras filas del dataframe para verificar que se ha cargado correctamente
df.head()


In [ ]:
# Informacion general del dataset
print(f"Filas: {df.shape[0]}, Columnas: {df.shape[1]}")
print()

# Tipo de dato de cada columna
print("Tipos de datos:")
for col, dtype in df.dtypes.items():
    print(f"  {col:20s} -> {dtype}")

print()

# Verificar valores faltantes
nulos = df.isna().sum()
if nulos.sum() == 0:
    print("No hay valores faltantes en el dataset.")
else:
    print(f"ATENCION: hay {nulos.sum()} valores nulos.")
    print(nulos[nulos > 0])

In [ ]:
df.describe().round(2) #Estadistica descriptiva de las columnas numéricas


In [ ]:
# DISTRIBUCION DE LA VARIABLE OBJETIVO
#
# Si el 99% de los clientes no abandonan, es un modelo que tendria una presicion (accuracy) completamente inutil
# porque siempre predicira el 99% de los datos, por lo tanto hay que balancear las clases
#

conteo = df["abandono"].value_counts()
porcentaje = df["abandono"].value_counts(normalize=True) * 100

print("Distribución de la variable objetivo 'abandono':")
print(f" Permanece (0) : {conteo[0]} clientes ({porcentaje[0]:.1f}%)")
print(f" Abandona (1)  : {conteo[1]} clientes ({porcentaje[1]:.1f}%)") 
print()
print("Las clases estan moderadamente desbalanceadas")
print("Hay mas clientes que permanecen que  clientes que abandonan")
print("Esto es lo tipico en la realidad: la mayoria de clientes NO se van.")

# Análisis exploratorio como se comportan las variables según abandono?

### antesde entrenar el modelo, vemos si las varaiables tienen relacion con el abandono. Esto nos ayuda  a entender que factores podrian influiren la decisiondel cliente

In [ ]:
#Compararel promedio de cada variable numerica segun si  el cliente abandono o no

comparacion = df.groupby("abandono")[["antiguedad_meses", "factura_mensual",
                                      "llamadas_soporte", "satisfaccion"]].mean().round(2)
comparacion.index = ["permanece (0)", "abandona (1)"]

print("Promedio de variable numerica segun abandono:")
print(comparacion)


In [ ]:
# Analisis del tipo de contrato vs abandono 

tabla_contrato = pd.crosstab(
    df["contrato"],
    df["abandono"],
    normalize="index"   
).round(3) * 100

tabla_contrato.columns = ["permanece (%)", "abandona (%)"]
print("porcentaje de abandono por tipo de contrato:")
print(tabla_contrato)

# Preparar los datos para el modelo

1. Seprar variables predictoras (X) de la variable objetivo.
2. Entrenar los datos con (80%) del dataset y probar con el (20%) del dataset

In [ ]:
# Paso 1: Separar X (variables independientes) e y (variable objetivo)

x = df[["antiguedad_meses", "factura_mensual", "llamadas_soporte", "satisfaccion", "contrato"]]

y = df["abandono"]

# Paso 2: Dividir en conjunto de entrenamiento (80%) y prueba (20%)
x_train, x_test, y_train, y_test = train_test_split(
    x, y, 
    test_size=0.2, # 20% para prueba
    random_state=42, # Semilla para reproducibilidad
    stratify=y
)

print(f" Datos de entrenamiento: {x_train.shape[0]} clientes")
print(f" Datos de prueba: {x_test.shape[0]} clientes")
print("")
print(f"Proporción de abandono en entrenamiento: {y_train.mean()*100:.1f}%")
print(f"Proporción de abandono en prueba: {y_test.mean()*100:.1f}%")
print("")
print(" Las proporciones son similares gracias al parámetro 'stratify=y' en train_test_split")


# Construir el pipeline y entrenar el modelo

1. calcula una puntuacion lineal
2. Pasa por esa puntuacion que la convierte en una probabilidad entre 0 y 1
3. Si la probabilidad es mayora 0.5, predice la clase 1 (abandono): si no, predice 0 (permanece)

In [ ]:
# Definir columna por tipo

from sklearn.pipeline import Pipeline


numeric_features = ["antiguedad_meses", "factura_mensual", "llamadas_soporte", "satisfaccion"]
categorical_features = ["contrato"]

# preprocesamineto
preprocess = ColumnTransformer(
    transformers=[
        ("num", "passthrough", numeric_features),
        ("cat", OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ]
)

#Pipeline: preprocesamiento + modelo
# Garantizar que los datos siempre se preprocesen antes de llegar al modelo
pipe = Pipeline(steps=[
    ("preprocess", preprocess),
    ("model", LogisticRegression(max_iter=1000))
])

print("Modelo entrenado correctamente")

# Hacer predicciones

Usamos el modelo entrenado para predecir el abandono en los datos de prueba.

In [ ]:
 #Generar predicciones conlo datos de prueba
y_pred = pipe.predict(x_test)

#Mostar las primeras 10 prediciones vs valores reales
comparacion_pred = pd.DataFrame({
    "Real": y_test.values[10],
    "Prediccion": y_pred[10],
    "Correcto" : ["Si" if r == p else "No" for r, p in zip(y_test.values[10], y_pred[10])]
})

print("Primeras 10 predicciones vs valores reales:")
print()

comparacion_pred

# Evaluacion del modelo: Exactitud/ Presicion (Acurracy)

cuando hablamos del porcentaje de precisión nos referimos al % de predicciones que fueron correctas

Accurracy = Predicciones correctas/ total de Predicciones

Ejemplo:  si el 95% de clienres **NO** abandona, un modelo  que siempre diga "No abandona" tendria un 95% de presición , pero jamas detectaria a un cliente en riesgo . Seria inutil

In [ ]:
acc = accuracy_score(y_test, y_pred)

print(f"Accuracy (exactitud): {acc:.3f} ({acc*100:.1f}%)")
print("")
print(f"Esto significa que el  modelo acertó {int(acc*len(y_test))} de {len(y_test)} predicciones.")

In [ ]:
# Calcular la matriz de confusión
cm = confusion_matrix(y_test, y_pred)

tn, fp, fn, tp = cm.ravel()

print("Matriz de Confusión:")
print(f"  Verdaderos Negativos (TN): {tn} — permanece y el modelo dijo permanece")
print(f"  Falsos Positivos    (FP): {fp} — permanece pero el modelo dijo abandona")
print(f"  Falsos Negativos    (FN): {fn} — abandona pero el modelo dijo permanece")
print(f"  Verdaderos Positivos(TP): {tp} — abandona y el modelo dijo abandona")
print()
print(f"Errores totales: {fp + fn} de {len(y_test)} predicciones")

In [ ]:
# Reporte de clasificación completo
print("Reporte de Clasificación:")
print()
print(classification_report(y_test, y_pred, 
                            target_names=["Permanece (0)", "Abandona (1)"]))

In [ ]:
import seaborn as sns

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# --- Gráfica 1: Matriz de Confusión como heatmap ---
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Permanece", "Abandona"],
            yticklabels=["Permanece", "Abandona"],
            ax=axes[0], cbar=False,
            annot_kws={"size": 16})
axes[0].set_xlabel("Predicción del modelo", fontsize=12)
axes[0].set_ylabel("Valor real", fontsize=12)
axes[0].set_title("Matriz de Confusión", fontsize=14, fontweight="bold")

# --- Gráfica 2: Importancia de las variables (coeficientes del modelo) ---
# Obtener nombres de las variables después del preprocesamiento
cat_names = pipe.named_steps["preprocess"] \
    .transformers_[1][1].get_feature_names_out(["contrato"]).tolist()
feature_names = numeric_features + cat_names

coefs = pipe.named_steps["model"].coef_[0]
coef_df = pd.DataFrame({"Variable": feature_names, "Coeficiente": coefs})
coef_df = coef_df.sort_values("Coeficiente")

colors = ["#2ecc71" if c < 0 else "#e74c3c" for c in coef_df["Coeficiente"]]
axes[1].barh(coef_df["Variable"], coef_df["Coeficiente"], color=colors)
axes[1].set_xlabel("Coeficiente (peso en el modelo)", fontsize=12)
axes[1].set_title("Importancia de Variables", fontsize=14, fontweight="bold")
axes[1].axvline(x=0, color="gray", linestyle="--", linewidth=0.8)

# Leyenda
axes[1].text(0.98, 0.02, "Rojo = aumenta abandono\nVerde = reduce abandono",
             transform=axes[1].transAxes, fontsize=9,
             ha="right", va="bottom",
             bbox=dict(boxstyle="round", facecolor="wheat", alpha=0.5))

plt.tight_layout()
plt.show()